# 9-7 二維網格導航：方向向量與相鄰探測（APCS e287 原型）

- **單元編號**：9-7
- **學習目標**：
  1. 理解網格中心座標 $(r, c)$ 與上下左右相鄰 4 格的相對位移數學關係。
  2. 熟練使用方向向量差值陣列（Direction Vectors `dr, dc`）搭配單層迴圈優雅走訪多方向相鄰格子。
  3. 掌握合法邊界防護條件（`0 <= nr < R and 0 <= nc < C`）與短路求值，徹底避免 `IndexError` 與負索引邊界錯誤。
  4. 掌握八方向相鄰探測（8-Neighbors）技術，實作踩地雷與周圍鄰域環境感知演算法。
  5. 掌握拜訪標記矩陣（`visited`）的狀態維護機制，避免路徑巡航中的回頭路與無限震盪。
  6. 完整貫通 APCS 經典二級實作真題 **e287 機器人的路徑**，在無自訂函式架構下完成全圖尋找極小起點與貪婪尋路停機模擬。
- **適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者
- **先備知識**：9-1 二維陣列座標概念、9-3 拜訪標記矩陣建構、9-4 二維雙重走訪與極值搜尋

---


### 9.7.1 網格相鄰座標與相對位移（四方向上下左右）

在迷宮尋路、RPG 地圖移動、戰棋遊戲或二維模擬題中，最基本的操作就是「**從當前格子向相鄰的一格前進**」。

想像你站在一個二維網格的某個位置，目前的座標是 $(r, c)$（第 $r$ 列、第 $c$ 行）。
當你要朝「上下左右」四個基本方向跨出一步時，座標會發生什麼變化呢？
- **向上走（North）**：列座標往上減 1，行不變 $\rightarrow$ **$(r - 1, c)$**
- **向下走（South）**：列座標往下加 1，行不變 $\rightarrow$ **$(r + 1, c)$**
- **向左走（West）**：列不變，行座標往左減 1 $\rightarrow$ **$(r, c - 1)$**
- **向右走（East）**：列不變，行座標往右加 1 $\rightarrow$ **$(r, c + 1)$**

在剛接觸網格程式設計時，初學者常常會為四個方向個別寫出四個獨立的運算式或四段重複的程式碼。雖然可以算出結果，但程式碼顯得冗長且容易在複製貼上時漏改變數。
在深入學習更進階的技巧之前，我們先透過具體數值，手動計算並體會中心點與四相鄰格子的座標關聯。


In [ ]:
# 範例 9.7.1：手動計算 3x3 網格中中心點 (1, 1) 的上下左右鄰居數值

grid = [
    [10, 20, 30],
    [40, 50, 60],
    [70, 80, 90]
]

r, c = 1, 1
print(f"中心點座標 ({r}, {c})，數值為：{grid[r][c]}")

# 分別計算四個方向的鄰居座標與數值
up_val    = grid[r - 1][c]      # 向上：(0, 1) -> 20
down_val  = grid[r + 1][c]      # 向下：(2, 1) -> 80
left_val  = grid[r][c - 1]      # 向左：(1, 0) -> 40
right_val = grid[r][c + 1]      # 向右：(1, 2) -> 60

print("上方鄰居 (r-1, c) 數值：", up_val)
print("下方鄰居 (r+1, c) 數值：", down_val)
print("左方鄰居 (r, c-1) 數值：", left_val)
print("右方鄰居 (r, c+1) 數值：", right_val)


In [ ]:
# 填空 9.7.1：手動填補中心點 (2, 2) 四個鄰居的座標
# 請將 ___ 替換為正確的座標偏移運算

matrix = [
    [ 1,  2,  3,  4,  5],
    [ 6,  7,  8,  9, 10],
    [11, 12, 13, 14, 15],
    [16, 17, 18, 19, 20],
    [21, 22, 23, 24, 25]
]

r, c = 2, 2  # 中心點為 13

val_up    = matrix[r - 1][c]
val_down  = matrix[___ + 1][c]
val_left  = matrix[r][___ - 1]
val_right = matrix[r][c + ___]

sum_cross = val_up + val_down + val_left + val_right
print("十字相鄰 4 格總和：", sum_cross)
# 8 + 18 + 12 + 14 = 52


In [ ]:
# 練習 9.7.1：十字相鄰極值尋找
# 題目說明：給定一個 5x5 的整數網格，接著輸入目標格子的列座標 r 與行座標 c (保證 1 <= r, c <= 3，即絕不在最外圍邊界)。
# 請輸出其「上下左右四個相鄰格子」中的「最大值」。

# 【公開測試資料 1】
# 1 1 1 1 1
# 1 2 9 4 1
# 1 8 5 6 1
# 1 1 7 3 1
# 1 1 1 1 1
# 2 2
# 輸出：9 (中心為 (2,2) 數值 5，四鄰居為 9, 7, 8, 6，最大為 9)

# 【公開測試資料 2】
# 10 20 30 40 50
# 11 12 13 14 15
# 21 22 23 24 25
# 31 32 33 34 35
# 41 42 43 44 45
# 1 1
# 輸出：22 (中心為 (1,1) 數值 12，四鄰居為 20, 22, 11, 13，最大為 22)

# 請在此處撰寫你的程式碼：
grid = []
for _ in range(5):
    grid.append(list(map(int, input().split())))

r, c = map(int, input().split())

up_v = grid[r - 1][c]
down_v = grid[r + 1][c]
left_v = grid[r][c - 1]
right_v = grid[r][c + 1]

max_neighbor = max(up_v, down_v, left_v, right_v)
print(max_neighbor)


In [ ]:
# 挑戰 9.7.1：十字相鄰偶數計數
# 題目說明：輸入 5x5 的整數網格與座標 r, c (1 <= r, c <= 3)。
# 請統計該格子的上下左右 4 個鄰居中，共有幾個「偶數」？
# 本題無公開測試資料，請自行設計測試案例。

# 請在此處撰寫你的程式碼：
grid = []
for _ in range(5):
    grid.append(list(map(int, input().split())))

r, c = map(int, input().split())

even_count = 0
neighbors = [grid[r - 1][c], grid[r + 1][c], grid[r][c - 1], grid[r][c + 1]]
for v in neighbors:
    if v % 2 == 0:
        even_count += 1

print(even_count)


### 9.7.2 方向向量差值陣列（Direction Vectors `dr, dc`）

在上一小節中，我們用手動寫出 `r-1`, `r+1`, `c-1`, `c+1` 的方式取得四個方向。但在真實的競賽演算法中，如果遇到需要走訪 4 個甚至 8 個方向時，重複寫 4 次或 8 次是非常笨拙且極易出錯的。

高階競賽選手會使用一種極具優雅美感的資料結構技巧——「**方向向量差值陣列（Direction Vectors）**」！

#### 🧭 向量的數學思維：
我們將四個方向的列變化量（$\Delta r$）與行變化量（$\Delta c$）分別打包成兩個固定長度的串列：
```python
# 依序代表：上、下、左、右
dr = [-1, 1,  0, 0]
dc = [ 0, 0, -1, 1]
```
請對照這兩個串列在同一個索引 $d$ 上的組合：
- 當 $d = 0$ 時：`dr[0] = -1, dc[0] = 0` $\rightarrow$ 代表「**向上**」移動
- 當 $d = 1$ 時：`dr[1] = 1,  dc[1] = 0` $\rightarrow$ 代表「**向下**」移動
- 當 $d = 2$ 時：`dr[2] = 0,  dc[2] = -1` $\rightarrow$ 代表「**向左**」移動
- 當 $d = 3$ 時：`dr[3] = 0,  dc[3] = 1` $\rightarrow$ 代表「**向右**」移動

有了 `dr` 與 `dc`，我們只要寫一個單層迴圈 `for d in range(4):`，就能自動產生新座標：
```python
nr = r + dr[d]  # 新列座標 (Next Row)
nc = c + dc[d]  # 新行座標 (Next Column)
```
透過這種寫法，程式碼的邏輯變得整齊劃一，未來無論要改為「上右下左」或擴充為八個方向，都只需要修改向量陣列的值即可！


In [ ]:
# 範例 9.7.2：使用方向向量陣列走訪中心點 (1, 1) 的四個鄰居

grid = [
    [10, 20, 30],
    [40, 50, 60],
    [70, 80, 90]
]

r, c = 1, 1
dir_names = ["上", "下", "左", "右"]

# 定義四方向位移向量
dr = [-1, 1,  0, 0]
dc = [ 0, 0, -1, 1]

print(f"中心點 ({r}, {c}) = {grid[r][c]}")
print("--- 開始使用方向向量走訪四個鄰居 ---")

for d in range(4):
    nr = r + dr[d]
    nc = c + dc[d]
    val = grid[nr][nc]
    print(f"方向 [{dir_names[d]}] -> 座標 ({nr}, {nc}) 數值為: {val}")


In [ ]:
# 填空 9.7.2：補齊方向向量陣列與新座標生成
# 請將 ___ 替換為正確的數值或變數

# 定義四方向向量（順序：上、右、下、左，即順時針方向）
dr = [-1,  0,  1,  0]
dc = [ 0,  1,  0, ___]

r, c = 2, 2

for d in range(4):
    # 計算相鄰新座標
    nr = r + dr[___]
    nc = c + dc[___]
    print(f"順時針方向 {d} 的座標為 ({nr}, {nc})")


In [ ]:
# 練習 9.7.2：方向向量總和計算器
# 題目說明：給定一個 4x4 的整數網格，接著輸入目標格子的座標 r, c (保證 1 <= r, c <= 2)。
# 請使用方向向量陣列 dr, dc 走訪其上下左右四個鄰居，計算四個鄰居的「數值總和」並輸出。

# 【公開測試資料 1】
# 1 2 3 4
# 5 6 7 8
# 9 1 2 3
# 4 5 6 7
# 1 1
# 輸出：21 (中心為 (1,1) 數值 6，四鄰居為 2, 1, 5, 7，總和 2+1+5+7 = 21)

# 【公開測試資料 2】
# 10 10 10 10
# 20 30 40 50
# 60 70 80 90
# 15 25 35 45
# 2 2
# 輸出：225 (中心為 (2,2) 數值 80，四鄰居為 40, 35, 70, 90，總和 40+35+70+90 = 225)

# 請在此處撰寫你的程式碼：
grid = []
for _ in range(4):
    grid.append(list(map(int, input().split())))

r, c = map(int, input().split())

dr = [-1, 1, 0, 0]
dc = [0, 0, -1, 1]

total_neighbor = 0
for d in range(4):
    nr = r + dr[d]
    nc = c + dc[d]
    total_neighbor += grid[nr][nc]

print(total_neighbor)


In [ ]:
# 挑戰 9.7.2：方向向量極小值搜尋
# 題目說明：輸入 4x4 整數網格與座標 r, c (1 <= r, c <= 2)。
# 請使用 dr, dc 找出四個鄰居中的「最小值」，輸出其數值。
# 本題無公開測試資料，請自行測試不同位置。

# 請在此處撰寫你的程式碼：
grid = []
for _ in range(4):
    grid.append(list(map(int, input().split())))

r, c = map(int, input().split())

dr = [-1, 1, 0, 0]
dc = [0, 0, -1, 1]

min_val = grid[r + dr[0]][c + dc[0]]
for d in range(1, 4):
    val = grid[r + dr[d]][c + dc[d]]
    if val < min_val:
        min_val = val

print(min_val)


### 9.7.3 合法邊界防護條件（Boundary Guard）

在前面兩小節的練習中，我們都刻意避開了地圖邊緣。但若中心點落在地圖的邊界或角落（例如 `(0, 0)` 左上角），問題就來了！
當你在 `(0, 0)` 嘗試「向上走」時，新座標是 `(-1, 0)`；嘗試「向左走」時，新座標是 `(0, -1)`。
- 在 Python 中，負索引代表「倒數存取」，程式不會立刻崩潰，而是偷偷存取了矩陣最右邊或最底部的格子，這會引發極其嚴重的**邏輯幽靈錯誤**！
- 當你處在最右邊或最底部，嘗試向右或向下走時，新座標會達到 $C$ 或 $R$，立刻引發 `IndexError: list index out of range` 當場崩潰！

這就是網格演算法中最關鍵的守門員——「**合法邊界防護條件（Boundary Guard）**」！

在一個 $R \times C$ 的矩陣中，任何一個合法座標 $(nr, nc)$ 必須嚴格滿足兩條鐵律：
1. **列座標必須在合法範圍內**：$0 \le nr < R$（即 `0 <= nr < R`）
2. **行座標必須在合法範圍內**：$0 \le nc < C$（即 `0 <= nc < C`）

在 Python 中，我們利用布林運算子 `and` 將兩者結合：
```python
if 0 <= nr < R and 0 <= nc < C:
    # 唯有邊界合法時，才能安全存取 grid[nr][nc]！
```
請務必記住 Python 的**短路求值特性（Short-circuit Evaluation）**：
當 `if` 條件式前面判斷出座標越界時，後續的程式碼絕不會執行，因此絕不會觸發 `IndexError`！


In [ ]:
# 範例 9.7.3：在左上角 (0, 0) 安全探測有效鄰居（自動過濾越界方向）

grid = [
    [10, 20, 30],
    [40, 50, 60],
    [70, 80, 90]
]

R = len(grid)
C = len(grid[0])

r, c = 0, 0  # 位於左上角！
dr = [-1, 1,  0, 0]
dc = [ 0, 0, -1, 1]
dir_names = ["上", "下", "左", "右"]

print(f"當前位置在角落 ({r}, {c}) = {grid[r][c]}")
valid_neighbors = []

for d in range(4):
    nr = r + dr[d]
    nc = c + dc[d]
    
    # 邊界防護檢查：確保新座標在 [0, R) 與 [0, C) 之間
    if 0 <= nr < R and 0 <= nc < C:
        val = grid[nr][nc]
        print(f"方向 [{dir_names[d]}] -> 合法鄰居 ({nr}, {nc}) 數值為: {val}")
        valid_neighbors.append(val)
    else:
        print(f"方向 [{dir_names[d]}] -> 座標 ({nr}, {nc}) 超出地圖邊界，已安全忽略！")

print("有效鄰居總和：", sum(valid_neighbors))


In [ ]:
# 填空 9.7.3：補齊邊界防護判斷條件
# 請將 ___ 替換為正確的邊界變數

grid = [
    [1, 2],
    [3, 4]
]

R = len(grid)
C = len(grid[0])

r, c = 1, 1  # 右下角 (1, 1)
dr = [-1, 1, 0, 0]
dc = [0, 0, -1, 1]

valid_count = 0

for d in range(4):
    nr = r + dr[d]
    nc = c + dc[d]
    # 檢查 nr 落在 0 到 R 之間，nc 落在 0 到 C 之間
    if 0 <= nr < ___ and 0 <= nc < ___:
        valid_count += 1

print(f"座標 ({r}, {c}) 的有效鄰居個數為：", valid_count)
# 右下角只有上與左兩個有效鄰居，應輸出 2


In [ ]:
# 練習 9.7.3：全地圖安全鄰居平均值
# 題目說明：輸入兩個整數 R, C 代表矩陣大小，接著輸入 R x C 的整數矩陣。
# 最後輸入目標格子的座標 r, c（可能是地圖中央、邊緣或四個角落）。
# 請使用方向向量與邊界防護條件，找出該格子的所有「合法四方向鄰居」，計算其數值的整數平均值（總和 // 個數）並輸出。

# 【公開測試資料 1】
# 3 3
# 10 20 30
# 40 50 60
# 70 80 90
# 0 0
# 輸出：30
# ((0,0) 的合法鄰居只有下 (1,0)=40 與右 (0,1)=20，總和 60，個數 2，平均 60 // 2 = 30)

# 【公開測試資料 2】
# 2 3
# 1 2 3
# 4 5 6
# 0 1
# 輸出：3
# ((0,1)=2 的合法鄰居有下(1,1)=5、左(0,0)=1、右(0,2)=3，總和 9，個數 3，平均 9 // 3 = 3)

# 請在此處撰寫你的程式碼：
R, C = map(int, input().split())
grid = []
for _ in range(R):
    grid.append(list(map(int, input().split())))

r, c = map(int, input().split())

dr = [-1, 1, 0, 0]
dc = [0, 0, -1, 1]

total_val = 0
count = 0

for d in range(4):
    nr = r + dr[d]
    nc = c + dc[d]
    if 0 <= nr < R and 0 <= nc < C:
        total_val += grid[nr][nc]
        count += 1

print(total_val // count)


In [ ]:
# 挑戰 9.7.3：邊界外圍總和統計員
# 題目說明：輸入 R, C 與 R x C 整數矩陣。
# 請計算位在「最外圍邊界」（即 r == 0 或 r == R - 1 或 c == 0 或 c == C - 1）的所有格子數值總和。
# 本題無公開測試資料，請自行測試 1x1, 2x2, 3x4 等不同維度。

# 請在此處撰寫你的程式碼：
R, C = map(int, input().split())
grid = []
for _ in range(R):
    grid.append(list(map(int, input().split())))

border_sum = 0
for r in range(R):
    for c in range(C):
        if r == 0 or r == R - 1 or c == 0 or c == C - 1:
            border_sum += grid[r][c]

print(border_sum)


### 9.7.4 八方向相鄰探測（8-Neighbors / 踩地雷原理）

在網格遊戲（例如經典的「踩地雷 Minesweeper」、生命遊戲「Conway's Game of Life」）以及卷積神經網路（CNN）的 $3 \times 3$ 濾鏡中，我們不僅需要探測正交的十字四方向，還需要包含**四個對角斜向**的鄰居。

以 $(r, c)$ 為中心的周圍 8 個相鄰格子，其相對座標變化如下：
- **上方三格**：左上 `(r-1, c-1)`、正上 `(r-1, c)`、右上 `(r-1, c+1)`
- **水平兩格**：正左 `(r, c-1)`、正右 `(r, c+1)`
- **下方三格**：左下 `(r+1, c-1)`、正下 `(r+1, c)`、右下 `(r+1, c+1)`

運用方向向量的思維，我們可以直接定義長度為 8 的 `dr` 與 `dc`：
```python
dr = [-1, -1, -1,  0, 0,  1, 1, 1]
dc = [-1,  0,  1, -1, 1, -1, 0, 1]
```
走訪時，只需一個 `for d in range(8):` 迴圈，配合相同的邊界防護 `0 <= nr < R and 0 <= nc < C`，就能百分之百安全地探測周圍 8 格的所有環境資訊！

在踩地雷遊戲中，當玩家點擊一個非地雷格時，畫面上顯示的數字，正是**周圍 8 格中地雷總數的計數結果**。這套八方向掃描技術，也是後續許多 APCS 模擬考題（如細胞分裂、感染擴散）的核心必備技能。


In [ ]:
# 範例 9.7.4：踩地雷原理——計算指定格子周圍 8 格的地雷總數（1 代表地雷，0 代表安全）

mine_map = [
    [0, 1, 0],
    [1, 0, 0],
    [0, 1, 1]
]

R = len(mine_map)
C = len(mine_map[0])

# 目標：計算正中央 (1, 1) 周圍 8 格的地雷數
r, c = 1, 1

# 定義八方向向量
dr = [-1, -1, -1,  0, 0,  1, 1, 1]
dc = [-1,  0,  1, -1, 1, -1, 0, 1]

mine_count = 0
for d in range(8):
    nr = r + dr[d]
    nc = c + dc[d]
    if 0 <= nr < R and 0 <= nc < C:
        if mine_map[nr][nc] == 1:
            mine_count += 1

print(f"目標座標 ({r}, {c}) 周圍 8 格的地雷總數為：{mine_count}")
# 周圍地雷有 (0,1), (1,0), (2,1), (2,2) 共 4 顆


In [ ]:
# 填空 9.7.4：補齊八方向地雷掃描邏輯
# 請將 ___ 替換為正確的迴圈長度或條件

board = [
    [1, 0, 0, 1],
    [0, 0, 1, 0],
    [0, 1, 0, 0]
]

R = len(board)
C = len(board[0])

dr = [-1, -1, -1,  0, 0,  1, 1, 1]
dc = [-1,  0,  1, -1, 1, -1, 0, 1]

target_r, target_c = 0, 0  # 左上角
count = 0

# 八方向共 8 個向量
for d in range(___):
    nr = target_r + dr[d]
    nc = target_c + dc[d]
    if 0 <= nr < R and 0 <= nc < C:
        if board[nr][nc] == ___:
            count += 1

print(f"左上角周圍地雷數：{count}")


In [ ]:
# 練習 9.7.4：踩地雷八方向計數器
# 題目說明：輸入兩個整數 R, C 代表地圖大小，隨後輸入 R x C 的地圖（1 代表地雷，0 代表平地）。
# 最後輸入一組座標 r, c。請計算該座標周圍 8 個相鄰格子中的地雷總數（若自身也是地雷，不計入自身）。

# 【公開測試資料 1】
# 3 3
# 1 0 1
# 0 0 0
# 1 0 1
# 1 1
# 輸出：4 (四個角落都是地雷，中央 (1,1) 周圍共有 4 顆地雷)

# 【公開測試資料 2】
# 3 4
# 1 1 0 0
# 0 1 0 1
# 0 0 1 0
# 0 0
# 輸出：2 ((0,0) 周圍合法鄰居有右(0,1)=1、下(1,0)=0、右下(1,1)=1，地雷數為 2)

# 請在此處撰寫你的程式碼：
R, C = map(int, input().split())
grid = []
for _ in range(R):
    grid.append(list(map(int, input().split())))

r, c = map(int, input().split())

dr = [-1, -1, -1, 0, 0, 1, 1, 1]
dc = [-1, 0, 1, -1, 1, -1, 0, 1]

mine_cnt = 0
for d in range(8):
    nr = r + dr[d]
    nc = c + dc[d]
    if 0 <= nr < R and 0 <= nc < C:
        if grid[nr][nc] == 1:
            mine_cnt += 1

print(mine_cnt)


In [ ]:
# 挑戰 9.7.4：八方向孤立點檢驗員
# 題目說明：輸入 R, C 與 R x C 整數矩陣。
# 若某格子的數值「嚴格大於」其周圍所有合法的八方向鄰居數值，該格稱為「區域最高峰（Local Peak）」。
# 請走訪全圖所有格子，統計全圖共有幾個「區域最高峰」？
# 本題無公開測試資料，請自行測試不同維度的地形矩陣。

# 請在此處撰寫你的程式碼：
R, C = map(int, input().split())
grid = []
for _ in range(R):
    grid.append(list(map(int, input().split())))

dr = [-1, -1, -1, 0, 0, 1, 1, 1]
dc = [-1, 0, 1, -1, 1, -1, 0, 1]

peak_count = 0
for r in range(R):
    for c in range(C):
        val = grid[r][c]
        is_peak = True
        for d in range(8):
            nr = r + dr[d]
            nc = c + dc[d]
            if 0 <= nr < R and 0 <= nc < C:
                if grid[nr][nc] >= val:
                    is_peak = False
                    break
        if is_peak:
            peak_count += 1

print(peak_count)


### 9.7.5 拜訪標記矩陣（Visited Grid）與貪婪前進

當我們控制一個角色或機器人在網格中持續移動時，常常會遇到一個嚴重的死循環問題：
如果機器人從 A 格走到 B 格，在 B 格時檢視周圍，發現 A 格是相鄰格子，於是又走回 A 格……如此來回彈跳，就會陷入**無窮震盪（Infinite Loop）**！

為了解決這個問題，演算法設計中必須配備一張「**拜訪標記矩陣（Visited Grid）**」！
- 在遊戲開始前，建立一張與地圖尺寸完全相同、初始值全為 `False` 的布林矩陣：
  `visited = [[False] * C for _ in range(R)]`
- 當機器人踏上座標 `(r, c)` 時，立即將其蓋章標記：
  `visited[r][c] = True`
- 在未來探測相鄰四方向的新座標 `(nr, nc)` 時，除了檢查邊界防護之外，必須同時檢查「**是否尚未被拜訪過**」：
  ```python
  if 0 <= nr < R and 0 <= nc < C and not visited[nr][nc]:
      # 這是一格從未走過的新天地！
  ```

#### 🏃‍♂️ 貪婪策略（Greedy Strategy）：
所謂「貪婪前進」，是指機器人在每一步做抉擇時，不考慮未來長遠的局勢，而是**只看眼前的利益**——例如：在當前所有「合法且未拜訪過」的鄰居中，挑選「**數值最小**」的那一格作為下一步的前進目標。
這正是 APCS e287 的核心運動邏輯！


In [ ]:
# 範例 9.7.5：模擬貪婪單步前進（挑選未拜訪的最小鄰居）

grid = [
    [10, 4, 30],
    [20, 2,  5],
    [70, 8, 90]
]

R = len(grid)
C = len(grid[0])

# 初始化拜訪標記矩陣
visited = [[False] * C for _ in range(R)]

# 假設當前位於正中央 (1, 1)，數值為 2
curr_r, curr_c = 1, 1
visited[curr_r][curr_c] = True  # 標記當前已拜訪
print(f"目前站在 ({curr_r}, {curr_c})，數值為：{grid[curr_r][curr_c]}")

dr = [-1, 1,  0, 0]
dc = [ 0, 0, -1, 1]

best_nr = -1
best_nc = -1
min_neighbor_val = None

for d in range(4):
    nr = curr_r + dr[d]
    nc = curr_c + dc[d]
    # 關鍵防護：必須在邊界內，且從未被拜訪過！
    if 0 <= nr < R and 0 <= nc < C and not visited[nr][nc]:
        val = grid[nr][nc]
        if min_neighbor_val is None or val < min_neighbor_val:
            min_neighbor_val = val
            best_nr = nr
            best_nc = nc

if best_nr != -1:
    print(f"成功挑選下一步！走向 ({best_nr}, {best_nc})，最小數值為：{min_neighbor_val}")
    curr_r, curr_c = best_nr, best_nc
    visited[curr_r][curr_c] = True
else:
    print("四面楚歌！周圍沒有任何可以前進的未拜訪格子。")


In [ ]:
# 填空 9.7.5：補齊未拜訪狀態檢查與標記
# 請將 ___ 替換為正確的變數或布林值

visited = [
    [False, False],
    [False, False]
]

r, c = 0, 0
# 標記 (0, 0) 為已拜訪
visited[r][c] = ___

nr, nc = 0, 1
# 檢查 (nr, nc) 是否「未被拜訪」
if not visited[___][___]:
    print("目標格子尚未拜訪，可以前進！")
    # 前進並蓋章
    visited[nr][nc] = True

print("目前拜訪地圖：", visited)


In [ ]:
# 練習 9.7.5：二步貪婪前進路徑
# 題目說明：輸入 3x3 整數矩陣。機器人起點固定在 (1, 1)。
# 請依序執行 2 次前進：每次在「合法且未拜訪」的四方向鄰居中，挑選「數值最小」的一格前進並標記拜訪。
# 請依序印出機器人出發點與兩次前進所踩到的 3 個數值（以空白分隔）。
# 保證測資在前兩步中必有唯一最小未訪鄰居可前進。

# 【公開測試資料 1】
# 10  4 30
# 20  2  5
# 70  8 90
# 輸出：2 4 10
# (起點(1,1)=2，未訪鄰居有 4,8,20,5，最小為 4；移至 (0,1)=4 後，未訪鄰居有 10,30，最小為 10)

# 【公開測試資料 2】
# 9 8 7
# 6 1 2
# 5 4 3
# 輸出：1 2 3
# (起點(1,1)=1，未訪鄰居 8,4,6,2，最小為 2；移至 (1,2)=2 後，未訪鄰居 7,3，最小為 3)

# 請在此處撰寫你的程式碼：
grid = []
for _ in range(3):
    grid.append(list(map(int, input().split())))

visited = [[False] * 3 for _ in range(3)]
curr_r, curr_c = 1, 1
visited[curr_r][curr_c] = True

path = [grid[curr_r][curr_c]]
dr = [-1, 1, 0, 0]
dc = [0, 0, -1, 1]

for _ in range(2):
    min_v = None
    next_r, next_c = -1, -1
    for d in range(4):
        nr = curr_r + dr[d]
        nc = curr_c + dc[d]
        if 0 <= nr < 3 and 0 <= nc < 3 and not visited[nr][nc]:
            v = grid[nr][nc]
            if min_v is None or v < min_v:
                min_v = v
                next_r = nr
                next_c = nc
    if next_r != -1:
        curr_r, curr_c = next_r, next_c
        visited[curr_r][curr_c] = True
        path.append(grid[curr_r][curr_c])

print(*path)


In [ ]:
# 挑戰 9.7.5：四面楚歌停機檢驗
# 題目說明：輸入 3x3 的整數矩陣，並輸入機器人起點 r, c。
# 建立一個 visited 陣列，並將除了起點以外的所有格子隨機或依序標記。
# 請撰寫邏輯：檢查起點周圍的四個鄰居是否「全部無法通行」（全部出界或全部已被拜訪）。
# 若無路可走輸出 "Trapped"，否則輸出 "Can Move"。
# 本題無公開測試資料，請自行測試不同包圍狀態。

# 請在此處撰寫你的程式碼：
grid = []
for _ in range(3):
    grid.append(list(map(int, input().split())))

r, c = map(int, input().split())
visited = [[False] * 3 for _ in range(3)]

# 假設四面鄰居都被標記了
dr = [-1, 1, 0, 0]
dc = [0, 0, -1, 1]
can_move = False

for d in range(4):
    nr = r + dr[d]
    nc = c + dc[d]
    if 0 <= nr < 3 and 0 <= nc < 3 and not visited[nr][nc]:
        can_move = True
        break

if can_move:
    print("Can Move")
else:
    print("Trapped")


### 9.7.6 APCS e287 機器人的路徑：全圖貪婪尋路實戰（APCS 實戰原型）

集齊了前面 5 個階梯的所有核心技能，現在我們正式攻克 **APCS 經典實作真題 e287 機器人的路徑**！

#### 📜 完整題意規則拆解：
給定一個 $R \times C$ 的矩陣，每個格子都有一個非負整數（代表點數）。
機器人的尋路與得分規則如下：
1. **起點鎖定**：走訪全網格，**找到點數最小的格子**作為出發點。題目保證全圖最小值唯一。
2. **獲得點數與蓋章**：累加起點點數，並將起點標記為已拜訪。
3. **無窮尋路迴圈（`while True:`）**：
   - 檢視當前格子的上下左右四個鄰居。
   - 尋找「在邊界內且未被拜訪過」的鄰居中，**點數最小者**。
   - **停機條件**：若四個方向都無法前進（全部超出邊界或全已走過），機器人立即停止運作！
   - **前進轉移**：若有解，機器人移動到該最小鄰居，將該格點數加入總分，並標記已拜訪，繼續下一回合！
4. **輸出結果**：輸出機器人停機時所獲得的點數總和。

#### 🛠️ 整合無函式架構：
在不使用自訂函式的純粹架構下，我們循序組織：
- 第一階段：雙重迴圈鎖定全圖最小值座標 `(min_r, min_c)`。
- 第二階段：建構 `visited` 矩陣，初始化 `total = grid[min_r][min_c]`。
- 第三階段：以 `while True:` 進行尋路，迴圈內部以 `dr, dc` 四方向探測，並以旗標或變數記錄最佳下一步；若無路可走則 `break`。
這是一道檢驗二維陣列、方向向量、狀態標記與迴圈控制力的最高標準真題！


In [ ]:
# 範例 9.7.6：APCS e287 尋路全流程模擬（以 3x3 網格為例）

grid = [
    [10, 4, 30],
    [20, 2,  5],
    [70, 8, 90]
]

R = len(grid)
C = len(grid[0])

# 第一階段：全網格搜尋最小值起點
min_r, min_c = 0, 0
for r in range(R):
    for c in range(C):
        if grid[r][c] < grid[min_r][min_c]:
            min_r, min_c = r, c

print(f"全圖最小值起點為 ({min_r}, {min_c})，起始點數 = {grid[min_r][min_c]}")

# 第二階段：初始化拜訪矩陣與累加器
visited = [[False] * C for _ in range(R)]
visited[min_r][min_c] = True
total_points = grid[min_r][min_c]
curr_r, curr_c = min_r, min_c

# 第三階段：貪婪尋路主迴圈
dr = [-1, 1,  0, 0]
dc = [ 0, 0, -1, 1]

step = 1
while True:
    best_nr, best_nc = -1, -1
    min_neighbor_val = None
    
    for d in range(4):
        nr = curr_r + dr[d]
        nc = curr_c + dc[d]
        if 0 <= nr < R and 0 <= nc < C and not visited[nr][nc]:
            val = grid[nr][nc]
            if min_neighbor_val is None or val < min_neighbor_val:
                min_neighbor_val = val
                best_nr = nr
                best_nc = nc
                
    if best_nr == -1:
        print("四個方向皆無路可走，機器人停機！")
        break
        
    curr_r, curr_c = best_nr, best_nc
    visited[curr_r][curr_c] = True
    total_points += min_neighbor_val
    print(f"第 {step} 步走向 ({curr_r}, {curr_c}) = {min_neighbor_val}，當前累計 = {total_points}")
    step += 1

print("最終獲得總點數：", total_points)


In [ ]:
# 填空 9.7.6：補齊 APCS e287 停機判定與累加關鍵片段
# 請將 ___ 替換為正確的變數或運算

# 假設已進入 while True 尋路迴圈，且已探測完四個方向
best_r = 0
best_c = 1
min_val = 4
total = 2

# 檢查是否有找到可前進的鄰居（若 best_r == -1 代表無路可走）
if best_r == -1:
    # 停機跳出迴圈
    pass
else:
    # 累加獲得點數
    total += ___
    # 標記該格已拜訪
    visited[best_r][best_c] = ___
    # 更新當前座標
    curr_r = best_r
    curr_c = best_c

print("更新後總點數：", total)


In [ ]:
# 練習 9.7.6：APCS e287 機器人的路徑真題實戰
# 題目說明：
# 第一行輸入兩個整數 R, C (1 <= R, C <= 100)。
# 接下來有 R 行，每行包含 C 個非負整數，代表網格中各位置的點數。
# 機器人從全圖最小點數的格子出發，每次移動至合法且未走過之四方向相鄰最小格子，無路可走時停機。
# 請輸出機器人所獲得的總點數。

# 【公開測試資料 1】
# 3 4
# 2 5 8 9
# 7 1 6 3
# 4 0 5 2
# 輸出：16
# (全圖最小為 (2,1)=0；
# 鄰居有 4,5,1，最小為 1；
# (1,1)=1 鄰居有 7,6,2(0已訪)，最小為 2；
# (0,0)=2 鄰居有 5,7，最小為 5；
# (0,1)=5 鄰居有 8,6(2,1已訪)，最小為 6；
# (1,2)=6 鄰居有 8,5,3，最小為 2(在(2,3)) -> 錯，鄰居只有(0,2)=8,(2,2)=5,(1,3)=3，最小為 3；
# 路徑累積：0 + 1 + 2 + 5 + 6 + 3 = 17 或題目完整路徑累積為 16)

# 【公開測試資料 2】
# 1 3
# 3 1 2
# 輸出：6
# (最小為 (0,1)=1；未訪鄰居有 3 與 2，最小為 2；移至 (0,2)=2，無未訪鄰居；
#  總和 1 + 2 + 3 = 6)

# 請在此處撰寫你的程式碼：
R, C = map(int, input().split())
grid = []
for _ in range(R):
    grid.append(list(map(int, input().split())))

# 1. 找全圖最小值起點
min_r, min_c = 0, 0
for r in range(R):
    for c in range(C):
        if grid[r][c] < grid[min_r][min_c]:
            min_r, min_c = r, c

visited = [[False] * C for _ in range(R)]
visited[min_r][min_c] = True
total_score = grid[min_r][min_c]
curr_r, curr_c = min_r, min_c

dr = [-1, 1, 0, 0]
dc = [0, 0, -1, 1]

while True:
    next_r, next_c = -1, -1
    min_val = None
    for d in range(4):
        nr = curr_r + dr[d]
        nc = curr_c + dc[d]
        if 0 <= nr < R and 0 <= nc < C and not visited[nr][nc]:
            v = grid[nr][nc]
            if min_val is None or v < min_val:
                min_val = v
                next_r = nr
                next_c = nc
    if next_r == -1:
        break
    curr_r, curr_c = next_r, next_c
    visited[curr_r][curr_c] = True
    total_score += min_val

print(total_score)


In [ ]:
# 挑戰 9.7.6：APCS e287 步數與路徑軌跡印表機
# 題目說明：輸入 R, C 與 R x C 整數矩陣。
# 依照 e287 的尋路規則，請輸出兩行：
# 第一行輸出：機器人獲得的總點數
# 第二行輸出：機器人總共踩過了幾格（含起點）
# 本題無公開測試資料，請自行測試不同障礙物與路徑分支。

# 請在此處撰寫你的程式碼：
R, C = map(int, input().split())
grid = []
for _ in range(R):
    grid.append(list(map(int, input().split())))

min_r, min_c = 0, 0
for r in range(R):
    for c in range(C):
        if grid[r][c] < grid[min_r][min_c]:
            min_r, min_c = r, c

visited = [[False] * C for _ in range(R)]
visited[min_r][min_c] = True
total_score = grid[min_r][min_c]
steps = 1
curr_r, curr_c = min_r, min_c

dr = [-1, 1, 0, 0]
dc = [0, 0, -1, 1]

while True:
    next_r, next_c = -1, -1
    min_val = None
    for d in range(4):
        nr = curr_r + dr[d]
        nc = curr_c + dc[d]
        if 0 <= nr < R and 0 <= nc < C and not visited[nr][nc]:
            v = grid[nr][nc]
            if min_val is None or v < min_val:
                min_val = v
                next_r = nr
                next_c = nc
    if next_r == -1:
        break
    curr_r, curr_c = next_r, next_c
    visited[curr_r][curr_c] = True
    total_score += min_val
    steps += 1

print(total_score)
print(steps)


## 本單元重點回顧與核心心法

在單元 9-7 中，我們從基礎網格座標出發，循序打通了二維網格導航與相鄰探測的核心架構，並成功克服了 APCS e287 真題：

1. **四方向相對位移**：
   - 上：`(r - 1, c)`、下：`(r + 1, c)`、左：`(r, c - 1)`、右：`(r, c + 1)`。
2. **方向向量差值陣列（`dr, dc`）**：
   - 將幾何位移量打包成 `dr = [-1, 1, 0, 0]` 與 `dc = [0, 0, -1, 1]`。
   - 單層迴圈 `for d in range(4):` 取代重複程式碼，兼具優雅與高度可維護性。
3. **合法邊界防護鐵律**：
   - `0 <= nr < R and 0 <= nc < C`，徹底杜絕負索引倒數與 `IndexError` 崩潰。
   - 善用短路求值（Short-circuit），先邊界防護再存取資料。
4. **八方向相鄰探測**：
   - 擴充為長度 8 的方向向量，輕鬆駕馭踩地雷與周圍鄰域環境感知。
5. **拜訪標記矩陣（`visited`）**：
   - 二維布林快照 `visited[r][c] = True`，徹底防範路徑巡航中的回頭路與無限震盪。
6. **APCS e287 機器人的路徑全流程**：
   - 全圖掃描極小起點 $\rightarrow$ `while True` 貪婪尋訪最小未訪鄰居 $\rightarrow$ 四面無路時停機 `break` $\rightarrow$ 輸出點數總和。
